# Handwriting → Text: training an OCR model from scratch

## Section 1 — Introduction

**Goal:** teach a neural network to read a picture of a handwritten line and type out the text.

```
[image of handwriting]  →  model  →  "It was a splendid interpretation of the"
```

The model reads **any** sentence, character by character. It does *not* contain a
list of known sentences or words, so it can write sentences it has never seen.
To prove that, we test it only on handwriting from **people and sentences that were
never used in training**.

**How to use this notebook:** click a cell, then press **Run** (▶) or `Shift + Enter`.
Run the cells **from top to bottom**. Each cell has an explanation above it.

| Section | What happens | Time (CPU) |
|---|---|---|
| 3 | download the dataset (~265 MB) | 1–3 min, only once |
| 5 | clean + preprocess images | ~2 min, only once |
| 8 | **train the model** | **hours** (skipped if a trained model already exists) |
| 10 | evaluate on the test set | ~2 min |
| 14 | fine-tune on new handwriting | ~10–15 min |

## Section 2 — Imports

We import the libraries we need. The project's own code lives in the `src/` folder;
this notebook mainly *calls* those functions so that the notebook and the command-line
scripts always do exactly the same thing.

* **TensorFlow / Keras** – the deep learning framework (builds and trains the network)
* **NumPy / pandas** – numbers and tables
* **Pillow (PIL)** – opening images
* **matplotlib** – plots

In [ ]:
import os, sys
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"          # hide TensorFlow's technical start-up messages

# Make the project's src/ folder importable (works no matter where the project is stored)
SRC = os.path.abspath(os.path.join(os.getcwd(), "..", "src")) if os.path.basename(os.getcwd()) == "notebooks" \
      else os.path.abspath("src")
sys.path.insert(0, SRC)

import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
import keras

import config                      # all paths and settings (src/config.py)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("Project folder:", config.PROJECT_DIR)
keras.utils.set_random_seed(config.SEED)  # same random numbers every time -> reproducible

## Section 3 — Download the dataset

**Dataset: IAM Handwriting Database (line level)**

| | |
|---|---|
| **Name** | IAM Handwriting Database – lines (`Teklia/IAM-line`) |
| **Source** | https://huggingface.co/datasets/Teklia/IAM-line (original: University of Bern, https://fki.tic.heia-fr.ch/databases/iam-handwriting-database) |
| **License** | listed as MIT on Hugging Face; the original IAM database is free for **non-commercial research**. OK for learning, not for selling a product. |
| **Samples** | 10,373 handwritten lines from ~650 writers |
| **Images** | grayscale scans of one line of English handwriting each, 128 px tall |
| **Labels** | the exact text of the line, e.g. `It was a splendid interpretation of the` |
| **Official split** | 6,482 train / 976 validation / 2,915 test – writers do not overlap |

Why this dataset? It is *the* standard benchmark for handwritten **sentence**
recognition (not just digits or single letters), it downloads automatically without
an account, and it is small enough for a normal laptop.

The files are saved to `data/raw/`. If they are already there, nothing is downloaded again.

In [ ]:
from download_data import download_dataset
raw_paths = download_dataset()
for split, path in raw_paths.items():
    print(f"{split:11s} {path.stat().st_size / 1e6:6.1f} MB  ->  {path}")

## Section 4 — Inspect the dataset

Before training anything, **look at your data**. We check:
* what a sample looks like (image + text)
* image sizes
* missing labels, corrupted images and duplicates

Each downloaded file is a table (a *parquet* file): one row per line, with the image (as
compressed bytes) and its text.

In [ ]:
from prepare_data import load_raw_split
raw = {split: load_raw_split(split) for split in config.DATASET_SPLITS}
for split, df in raw.items():
    print(f"{split:11s}: {len(df):5d} lines")
raw["train"].head()

In [ ]:
# Show some handwritten lines with their text:  Image -> "text"
fig, axes = plt.subplots(6, 1, figsize=(14, 9))
for ax, (_, row) in zip(axes, raw["train"].sample(6, random_state=0).iterrows()):
    ax.imshow(Image.open(io.BytesIO(row["image_bytes"])), cmap="gray")
    ax.set_title(f'→ "{row["text"]}"', loc="left", fontsize=11)
    ax.axis("off")
plt.tight_layout(); plt.show()

Notice the IAM labelling style: punctuation is separated by spaces (`part . The`,
`B B C's`). The model learns to write it that way too.

In [ ]:
# Image sizes and text lengths
sizes = np.array([Image.open(io.BytesIO(b)).size for b in raw["train"]["image_bytes"]])
lengths = raw["train"]["text"].str.len()
print(f"image height: {sizes[:, 1].min()}–{sizes[:, 1].max()} px")
print(f"image width : {sizes[:, 0].min()}–{sizes[:, 0].max()} px (average {sizes[:, 0].mean():.0f})")
print(f"text length : {lengths.min()}–{lengths.max()} characters (average {lengths.mean():.0f})")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3))
a1.hist(sizes[:, 0], bins=50); a1.set_title("image widths (px)")
a2.hist(lengths, bins=40, color="tab:orange"); a2.set_title("text lengths (characters)")
plt.show()

In [ ]:
# Data quality checks
import hashlib
for split, df in raw.items():
    missing = (df["text"].isna() | (df["text"].str.strip() == "")).sum()
    corrupted = 0
    for b in df["image_bytes"]:
        try:
            Image.open(io.BytesIO(b)).load()
        except Exception:
            corrupted += 1
    duplicates = pd.Series([hashlib.md5(b).hexdigest() for b in df["image_bytes"]]).duplicated().sum()
    print(f"{split:11s}: missing labels {missing}, corrupted images {corrupted}, duplicate images {duplicates}")

train_sentences = set(raw["train"]["text"])
print("\nTest lines whose exact sentence also appears in training:",
      raw["test"]["text"].isin(train_sentences).sum(), "(they will be removed from training)")

## Section 5 — Prepare the data

### Train / validation / test split
IAM comes with an **official split**, so we use it instead of splitting randomly.
It is *writer-independent*: nobody who wrote a test line wrote any training line.

* **Training set** (~6,476 lines) – the model **learns** from these examples.
* **Validation set** (976 lines) – checked after every epoch to see whether the model
  generalises to new writers. We use it to pick the best model and to stop training.
* **Test set** (2,915 lines) – used **only once, at the very end**, for the final score.
  It is never used for any decision, so its score is an honest estimate of real performance.

Extra safety: the few training lines whose sentence also appears in validation/test are
removed, so **every test sentence is new** to the model.

### Image preprocessing (and why)
1. **Grayscale** – colour doesn't tell you which letter it is.
2. **Auto-contrast** – faint pencil and dark ink end up looking alike.
3. **Crop empty margins** – important for photos with lots of empty paper.
4. **Resize to 64 px height, keeping the aspect ratio** – the model needs a fixed height, but
   squashing or stretching would distort the letters. Long lines simply become wide images.
5. **Invert + normalise to 0..1** – ink = 1, paper = 0 (networks learn better on small numbers,
   and padding with 0 then means "nothing here").
6. **Padding + batching** – a batch must have one shape, so shorter lines get empty space
   on the right. Lines of similar width are grouped together, so there is little padding.

The prepared images go to `data/train/`, `data/validation/`, `data/test/` (PNG + `labels.csv`).

In [ ]:
from prepare_data import prepare_dataset, load_labels
report = prepare_dataset()          # takes ~2 minutes the first time, then it is skipped
pd.DataFrame(report).T

In [ ]:
train_df, val_df, test_df = load_labels("train"), load_labels("validation"), load_labels("test")
total = len(train_df) + len(val_df) + len(test_df)
for name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(f"{name:11s}: {len(df):5d} lines ({len(df) / total:.0%})")
print("\nSentences shared between train and test:", len(set(train_df.text) & set(test_df.text)))
train_df.head()

In [ ]:
# What the model actually receives: preprocessed image, and some augmented versions
from preprocessing import to_model_input
from dataset import _load_png, augment

example = train_df.iloc[5]
img = Image.open(example["path"])
fig, axes = plt.subplots(5, 1, figsize=(14, 7))
axes[0].imshow(img, cmap="gray"); axes[0].set_title(f"preprocessed (64 px tall): {example['text']}", loc="left")
axes[1].imshow(to_model_input(np.asarray(img))[..., 0], cmap="gray")
axes[1].set_title("model input: inverted, values 0..1 (ink = bright)", loc="left")
x = _load_png(example["path"])
for i, ax in enumerate(axes[2:]):
    ax.imshow(augment(x)[..., 0], cmap="gray")
    ax.set_title("random augmentation (training only: slant / width / pen thickness / ink)", loc="left")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

**Augmentation** = small random changes to training images (slant, width, stroke thickness,
ink intensity) so the model sees a slightly different version every epoch. This helps it
handle writing styles it has never seen. It is **only** used for training, never for
validation or test.

## Section 6 — Build the vocabulary

A neural network only understands numbers. So every character gets an ID:

```
0 → [BLANK]   (special CTC symbol: "no character here")
1 → [UNK]     (any character the model has never seen)
2 → ' ' (space)
3 → '!'
...
```

The vocabulary is built **only from the training labels**. Looking at the test labels here
would leak information about the test set.

In [ ]:
from vocab import Vocabulary
vocab = Vocabulary.from_texts(train_df["text"])
print(vocab)
print("characters:", "".join(vocab.characters))
print("\ncharacter → integer:", {c: vocab.char_to_id[c] for c in ["a", "b", "A", "0", " ", "."]})
print("integer → character:", {i: vocab.id_to_char[i] for i in range(6)})

sentence = "Hello world"
ids = vocab.encode(sentence)
print(f"\n{sentence!r} → {ids} → {vocab.decode(ids)!r}")
print("unknown character:", repr("café"), "→", vocab.encode("café"), "→", repr(vocab.decode(vocab.encode("café"))))

## Section 7 — Build the model (CNN + Bidirectional LSTM + CTC)

```
image 64 × W
   │  CNN: 5 convolution blocks
   │       learns visual features: strokes, curves, loops, parts of letters.
   │       Pooling shrinks the image to 4 × W/4.
   ▼
sequence of W/4 feature vectors   (each column of the image = one step, read left → right)
   │  2 × Bidirectional LSTM
   │       reads the sequence in both directions, so each position knows its
   │       neighbours (needed to tell "rn" from "m", or "cl" from "d").
   ▼
Dense + softmax: for every step, a probability for each character + BLANK
   │  CTC decoding: best character per step → merge repeats → remove blanks
   ▼
"hello"
```

### Why CTC?
For training we only know *which* text is in the image, not *where* each letter is.
**CTC (Connectionist Temporal Classification)** solves this: the loss adds up the probability of
**all** ways the characters could be spread over the time steps. Example with 10 steps:

```
h h - e - l l - l o     →  merge repeats: h - e - l - l o  →  remove blanks (-): "hello"
```
The blank between the two `l`s is what allows double letters.

In [ ]:
from model import build_model, compile_model
model = compile_model(build_model(vocab.size), config.LEARNING_RATE)
model.summary()

In [ ]:
# Before training the output is random. Quick look at the shapes:
dummy = np.zeros((1, config.IMG_HEIGHT, 1000, 1), dtype="float32")
print("input :", dummy.shape, " (batch, height, width, channels)")
print("output:", model.predict(dummy, verbose=0).shape, " (batch, time steps = 1000/4, classes)")

## Section 8 — Train

**Training** = show the model batches of 16 images, measure how wrong it is (the CTC
**loss**) and adjust its ~1 million weights a tiny bit to be less wrong. One pass over all
training images is an **epoch**.

Callbacks (helpers that run after every epoch):
* **CER callback** – reads the validation set and prints `Actual` vs `Predicted`, plus the
  validation character error rate. Loss alone is hard to interpret.
* **ModelCheckpoint** – saves the model whenever the validation CER is the best so far →
  `models/handwriting_ocr.keras`
* **ReduceLROnPlateau** – halves the learning rate when progress stalls (finer steps).
* **EarlyStopping** – stops when the validation CER hasn't improved for 10 epochs.

**On a laptop CPU one epoch takes about 20 minutes**, so a good model needs several hours.
Therefore:
* If a trained model already exists, this cell **skips training** and you can continue.
* Set `TRAIN = True` to train anyway. You can also run training in a terminal
  (`python src/train.py`) and continue this notebook afterwards.
* `QUICK_TEST = True` trains on 300 lines for 2 epochs (~2 min) just to see that everything
  works – the result will **not** read anything yet.

In [ ]:
from train import train, save_loss_plot

TRAIN = not config.MODEL_PATH.exists()   # train only if there is no trained model yet
QUICK_TEST = False                       # True = tiny 2-minute test run (useless model!)
EPOCHS = 20

if TRAIN:
    if QUICK_TEST:
        history = train(epochs=2, limit=300, verbose=1)
    else:
        history = train(epochs=EPOCHS, verbose=1)
else:
    print("A trained model already exists:", config.MODEL_PATH)
    print("Skipping training. (Set TRAIN = True to train again.)")

In [ ]:
# Training curves (from results/training_log.csv)
log = pd.read_csv(config.RESULTS_DIR / "training_log.csv")
save_loss_plot()
print(log[["epoch", "loss", "val_loss", "val_cer", "val_wer"]].round(3).to_string(index=False))
Image.open(config.RESULTS_DIR / "loss_curve.png")

How to read the curves: both losses should go **down**. If the training loss keeps falling
but the validation loss goes up, the model is **memorising** the training images
(*overfitting*) – that's when EarlyStopping helps.

## Section 9 — View predictions

We load the **best** saved model and let it read some validation lines.
(The model never trained on these – but we did use them to choose the best epoch, so the
honest final number comes from the test set in the next section.)

In [ ]:
from model import load_trained_model, predict_texts
from preprocessing import load_image

model = load_trained_model(config.MODEL_PATH)
vocab = Vocabulary.load(config.VOCAB_PATH)

sample = val_df.sample(6, random_state=7)
images = [to_model_input(load_image(p)) for p in sample["path"]]
results = predict_texts(model, images, vocab)

fig, axes = plt.subplots(len(sample), 1, figsize=(14, 2 * len(sample)))
for ax, (_, row), (text, conf) in zip(axes, sample.iterrows(), results):
    ax.imshow(Image.open(row["path"]), cmap="gray")
    ax.set_title(f"Actual:    {row['text']}\nPredicted: {text}   (confidence {conf:.0%})",
                 loc="left", family="monospace", fontsize=10)
    ax.axis("off")
plt.tight_layout(); plt.show()

## Section 10 — Evaluate on the test set (CER and WER)

Plain "accuracy" (is the whole line exactly right?) is too strict for OCR: one wrong letter
would make the whole line count as a failure. Instead we use:

* **CER – Character Error Rate:** how many character edits (wrong / missing / extra
  characters) are needed to fix the prediction, divided by the number of characters.
* **WER – Word Error Rate:** the same, counted in whole words.

```
Actual:     hello world        (11 characters, 2 words)
Predicted:  helo world
CER = 1 missing "l" / 11 characters = 0.09  (9 %)
WER = 1 wrong word  / 2 words       = 0.50  (50 %)
```
Lower is better; 0 is perfect. WER is always much higher than CER, because a single wrong
letter makes the whole word wrong.

In [ ]:
from metrics import cer, wer
print("CER:", round(cer("hello world", "helo world"), 3), "| WER:", wer("hello world", "helo world"))

In [ ]:
import matplotlib
from evaluate import evaluate_split, summarize, show_examples, save_error_analysis

test_results = evaluate_split(model, vocab, "test")      # ~2 minutes for 2,915 lines
metrics = summarize(test_results)
print(f"TEST  CER = {metrics['CER']:.3f}   WER = {metrics['WER']:.3f}   "
      f"perfect lines: {metrics['perfect_lines']} / {metrics['lines']}")

In [ ]:
# Good AND bad examples - all of these sentences were never seen during training
show_examples(test_results, config.RESULTS_DIR / "example_predictions.png", n_good=4, n_bad=4)
plt.show()

In [ ]:
print(save_error_analysis(test_results))

Typical weaknesses you will see: similar-looking letters (`a`/`o`, `e`/`c`, `n`/`u`, `r`/`n`),
punctuation and capital letters, missing spaces, and unusual or messy handwriting. A bigger
model, longer training (GPU) and a language model (dictionary) would reduce these errors.

## Section 11 — Save the model

The best model was already saved during training by `ModelCheckpoint`:
`models/handwriting_ocr.keras`.

A **`.keras` file** is one zip file that contains everything: the architecture (layers),
the learned weights (~1 million numbers) and the training settings. That's why you never
have to train from scratch again – training *produces* this file, and loading it gives
you the trained model back in seconds.

`models/vocab.json` (character ↔ number table) must stay next to it, because the model's
output numbers mean nothing without it.

In [ ]:
model.save(config.MODEL_PATH)            # (re)save the best model - this is how you save any Keras model
print(config.MODEL_PATH, f"{config.MODEL_PATH.stat().st_size / 1e6:.1f} MB")
print(config.VOCAB_PATH, f"{config.VOCAB_PATH.stat().st_size / 1e3:.1f} KB")

## Section 12 — Load the model again (no training needed)

Our model uses a **custom loss function** (`ctc_loss`). Keras must know that function
when it loads the file, so we pass it in `custom_objects`. That is the only special step.
(If you only want to *predict*, `compile=False` also works and skips the loss entirely.)

In [ ]:
from model import ctc_loss

loaded_model = keras.models.load_model(config.MODEL_PATH, custom_objects={"ctc_loss": ctc_loss})

# Check: the loaded model gives exactly the same output as the one in memory
x = to_model_input(load_image(test_df["path"].iloc[0]))[np.newaxis]
same = np.allclose(model.predict(x, verbose=0), loaded_model.predict(x, verbose=0), atol=1e-5)
print("Loaded model gives identical predictions:", same)
print("Prediction:", predict_texts(loaded_model, [x[0]], vocab)[0][0])
print("Actual:    ", test_df["text"].iloc[0])

## Section 13 — Test your own handwriting

1. Write one or a few lines on **white, unlined paper** with a **dark pen**.
2. Take a photo in good light (no shadows), or scan it.
3. **Crop** the photo so it only contains the handwriting.
4. Save it, e.g. as `data/my_images/my_note.jpg`, and put its path in `MY_IMAGE` below.

Several lines in one image are split automatically. If no image of yours is found, the cell
uses an **original** (un-preprocessed) test image instead so you can still see how it works.

What to expect: the model learned from ~650 people writing on scanned forms. **Your**
handwriting and phone photos are new for it, so expect more mistakes than on the test set.
Fine-tuning (next section) is the fix.

In [ ]:
from predict import recognize, confidence_label

MY_IMAGE = config.PROJECT_DIR / "data" / "my_images" / "my_note.jpg"   # <- change this path

if not MY_IMAGE.exists():
    print(f"{MY_IMAGE} not found - using an original test image as an example instead.\n")
    MY_IMAGE = config.RESULTS_DIR / "example_input.png"
    first = load_raw_split("test").iloc[10]
    Image.open(io.BytesIO(first["image_bytes"])).save(MY_IMAGE)
    print("Correct text:", first["text"])

results = recognize(MY_IMAGE, loaded_model, vocab)
plt.figure(figsize=(14, 3)); plt.imshow(Image.open(MY_IMAGE), cmap="gray"); plt.axis("off"); plt.show()
print("Recognized text:")
for text, conf in results:
    print(" ", text, f"   (confidence {conf:.0%}, {confidence_label(conf)})")

## Section 14 — Fine-tuning on new handwriting

```
Existing trained OCR model → load → add new handwriting → continue training with a
small learning rate → evaluate → save improved model (separately)
```

**When is fine-tuning useful?** When the model reads *your* handwriting (or your kind of
images – phone photos, pencil, lined paper) badly. Because the model already knows what
letters look like, a **small** amount of new data (50–200 lines) is enough. Training from
scratch would need thousands of lines and hours.

Key points:
* **Small learning rate** (1e-4 instead of 1e-3): only nudge the weights, don't destroy them.
* **Replay**: some original IAM lines are mixed in so the model doesn't forget them.
* The original model is kept; the new one is saved as `models/handwriting_ocr_finetuned.keras`.

### Using your own handwriting
Create `data/my_handwriting/` with your line images and a `labels.csv`:
```
file,text
line001.jpg,The weather was beautiful today.
line002.jpg,I went to the market yesterday
```

### Demo (if you have no data yet)
The cell below then creates `data/finetune_demo/`: IAM lines restyled to look like a different
"writer + camera" (strong slant, thick blurry strokes, gray noisy paper). The original model
struggles with this style; fine-tuning should fix much of that. (It uses only train and
validation lines - the test set stays untouched.)

In [ ]:
from finetune import finetune, create_demo_dataset, MY_DATA_DIR

if (MY_DATA_DIR / "labels.csv").exists():
    data_folder = MY_DATA_DIR
    print("Using YOUR handwriting from", data_folder)
else:
    data_folder = create_demo_dataset()
    print("No data/my_handwriting/labels.csv yet - using the demo dataset.")

# Show a few of the new examples
demo_labels = pd.read_csv(data_folder / "labels.csv", dtype=str, keep_default_na=False)
fig, axes = plt.subplots(3, 1, figsize=(14, 4))
for ax, (_, row) in zip(axes, demo_labels.head(3).iterrows()):
    ax.imshow(Image.open(data_folder / row["file"]), cmap="gray"); ax.set_title(row["text"], loc="left"); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ~10-15 minutes on a CPU
result = finetune(data_folder, epochs=8)
pd.DataFrame({"before": [result["before_new_cer"], result["before_iam_cer"]],
              "after":  [result["after_new_cer"],  result["after_iam_cer"]]},
             index=["CER on new handwriting", "CER on original IAM lines"]).round(3)

In [ ]:
# Load the fine-tuned model like any other saved model
finetuned = keras.models.load_model(config.FINETUNED_MODEL_PATH, custom_objects={"ctc_loss": ctc_loss})
print("Fine-tuned model loaded from", config.FINETUNED_MODEL_PATH)

## Done!

You have downloaded a dataset, prepared it, trained a CNN + BiLSTM + CTC handwriting
recognizer, evaluated it honestly with CER/WER, saved and loaded it, and fine-tuned it.

From a terminal (in the project folder) you can now simply run:
```
python src/predict.py path/to/your_handwriting.jpg
```